# 01_Exploration_Ref — Referenzbereich-Nachrechnung

**Frage:** `QSErgBewStrukDialog` liefert IQTIGs bereits fertige Bewertung (auffaellig/nicht auffaellig).
`QS.Qualitaetsindikator.csv` enthaelt aber zusaetzlich die Rohgroessen, aus denen diese Bewertung
eigentlich hervorgeht: `QSQI.Ergebnis` (gemessener Wert), `QSQI.Referenzwert` (Schwellenwert) und
`QSQI.Operator` (Vergleichsrichtung, z. B. `<=`). Was passiert, wenn wir die Auffaelligkeit selbst
aus diesen drei Spalten nachrechnen, statt `QSErgBewStrukDialog` zu uebernehmen — kommen wir auf
dasselbe Ergebnis?

**Kurzfassung des Ergebnisses:** Nein, nicht ganz — und der Grund dafuer ist selbst ein interessanter
Befund. Dieses Notebook ist eine reine Kontrollrechnung / ein Exkurs; **es aendert nichts an der
Analysetabelle oder der im Hauptprojekt verwendeten Ziel-Variable.**

In [1]:
import pandas as pd
import numpy as np

PFAD = "../Data/CSV/QS.Qualitätsindikator.csv"
qi = pd.read_csv(PFAD, low_memory=False)
print(f"Rohzeilen: {len(qi):,}")
print(f"Spalten mit Bezug zum Referenzbereich: "
      f"{[c for c in qi.columns if 'Referenz' in c or 'Operator' in c or 'Ergebnis' in c or 'Vertrauensbereich' in c]}")

Rohzeilen: 417,799
Spalten mit Bezug zum Referenzbereich: ['QSQI.BezugAndereQSErgebnisse', 'QSQI.BundVertrauensbereich', 'QSQI.Ergebnis', 'QSQI.ErgebnisMehrfach', 'QSQI.KHVertrauensbereich', 'QSQI.Operator', 'QSQI.Referenzbereich', 'QSQI.Referenzwert']


## 1. Nur echte Qualitätsindikatoren betrachten

Wie im Hauptnotebook: Zählkennzahlen (`EKez`/`TKez`/`KKez`) sind keine echten Bewertungen und werden
ausgeschlossen.

In [2]:
qi_qi = qi[qi["QSQI.ArtDesWertes"] == "QI"].copy()
print(f"Nach QI-Filter: {len(qi_qi):,} Zeilen")

Nach QI-Filter: 308,726 Zeilen


## 2. Die drei Rohspalten im Detail

**`QSQI.Operator`** — bei rund einem Drittel der Zeilen fehlt er komplett (dort gibt es gar keinen
Schwellenwert-Vergleich, siehe unten). Zwei Werte kommen vor: `<=` (Ergebnis soll unter dem
Referenzwert bleiben) und `>=` (Ergebnis soll über dem Referenzwert liegen).

In [3]:
print(qi_qi["QSQI.Operator"].value_counts(dropna=False))
print()
print(f"Anteil mit Operator vorhanden: {qi_qi['QSQI.Operator'].notna().mean():.1%}")

QSQI.Operator
<=     174872
NaN    100882
>=      32972
Name: count, dtype: int64

Anteil mit Operator vorhanden: 67.3%


**`QSQI.Ergebnis`** — der gemessene Wert. Meist eine normale Zahl (Dezimalpunkt, kein Komma),
aber bei **kleinen Fallzahlen maskiert IQTIG den echten Wert aus Datenschutzgründen** mit dem Text
`<=3` — der exakte Wert bleibt damit unbekannt, wir wissen nur „höchstens 3".

In [4]:
qi_qi["ergebnis_maskiert"] = qi_qi["QSQI.Ergebnis"].astype(str) == "<=3"
print(f"Anteil maskierter Ergebnisse (\'<=3\'): {qi_qi['ergebnis_maskiert'].mean():.1%}")
print(f"Anzahl maskierter Zeilen: {qi_qi['ergebnis_maskiert'].sum():,}")

Anteil maskierter Ergebnisse ('<=3'): 23.0%
Anzahl maskierter Zeilen: 70,915


**`QSQI.Referenzwert`** — der Schwellenwert selbst, im deutschen Komma-Format (z. B. `4,18`),
muss vor der Umrechnung erst in einen Dezimalpunkt umgewandelt werden — derselbe Formatfehler-Fallstrick
wie bei `FA.Personal.Anzahl` im Hauptnotebook.

In [5]:
qi_qi["ergebnis_num"] = pd.to_numeric(qi_qi["QSQI.Ergebnis"], errors="coerce")
qi_qi["referenzwert_num"] = pd.to_numeric(
    qi_qi["QSQI.Referenzwert"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
)
print(qi_qi[["QSQI.Ergebnis", "ergebnis_num", "QSQI.Referenzwert", "referenzwert_num"]].sample(5, random_state=1))

       QSQI.Ergebnis  ergebnis_num QSQI.Referenzwert  referenzwert_num
19498          78.93         78.93               NaN               NaN
98688              0          0.00              0,14              0.14
139838          1.13          1.13              2,52              2.52
246780          0.33          0.33              2,79              2.79
268452           0.4          0.40              2,78              2.78


## 3. Wie viele Zeilen sind überhaupt selbst berechenbar?

Damit wir selbst eine Auffälligkeit berechnen können, brauchen wir gleichzeitig: einen echten
Zahlenwert bei `QSQI.Ergebnis` (nicht maskiert), einen `QSQI.Referenzwert` und einen `QSQI.Operator`.

In [6]:
berechenbar = (
    qi_qi["ergebnis_num"].notna()
    & qi_qi["referenzwert_num"].notna()
    & qi_qi["QSQI.Operator"].notna()
)
print(f"Selbst berechenbare Zeilen: {berechenbar.sum():,} von {len(qi_qi):,} ({berechenbar.mean():.1%})")

qi_calc = qi_qi[berechenbar].copy()

Selbst berechenbare Zeilen: 111,061 von 308,726 (36.0%)


In [7]:
schritt0 = len(qi_qi)
schritt1 = qi_qi[qi_qi["QSQI.Operator"].notna()]
schritt2 = schritt1[schritt1["ergebnis_num"].notna()]

print(f"0. Ausgangsdatensatz (nach QI-Filter): {schritt0:,}")
print(f"1. QSQI.Operator vorhanden: {len(schritt1):,} (entfernt: {schritt0-len(schritt1):,})")
print(f"2. QSQI.Ergebnis ist echte Zahl (nicht maskiert): {len(schritt2):,} (entfernt: {len(schritt1)-len(schritt2):,})")
print()
print(f"Verbleibende Zeilen ohne Referenzwert: {schritt2['referenzwert_num'].isna().sum()}")

0. Ausgangsdatensatz (nach QI-Filter): 308,726
1. QSQI.Operator vorhanden: 207,844 (entfernt: 100,882)
2. QSQI.Ergebnis ist echte Zahl (nicht maskiert): 111,061 (entfernt: 96,783)

Verbleibende Zeilen ohne Referenzwert: 0


Nur **36 %** der QI-Zeilen lassen sich überhaupt selbst nachrechnen — bei den übrigen 64 % fehlt entweder der Referenzwert/Operator komplett (bei vielen N*-codierten Zeilen ohnehin erwartbar, aber auch bei einem Teil der offiziell bewerteten Zeilen, siehe Abschnitt 5) oder das Ergebnis ist maskiert. QSQI.Referenzwert ist bei der nach Schritt 2 verbleibenden Menge in jedem Fall bereits vorhanden — die drei Spalten werden von IQTIG offenbar immer gemeinsam befüllt oder gemeinsam leer gelassen.

## 4. Eigene Bewertung berechnen und mit `QSErgBewStrukDialog` vergleichen

Regel: Bei Operator `<=` ist ein Haus unauffällig, wenn `Ergebnis <= Referenzwert`; bei `>=`
unauffällig, wenn `Ergebnis >= Referenzwert`. Verglichen wird nur mit Zeilen, die auch offiziell
bewertet sind (kein N*-Code).

In [8]:
def eigene_bewertung(row):
    if row["QSQI.Operator"] == "<=":
        return 0 if row["ergebnis_num"] <= row["referenzwert_num"] else 1
    elif row["QSQI.Operator"] == ">=":
        return 0 if row["ergebnis_num"] >= row["referenzwert_num"] else 1
    return np.nan

qi_calc["eigene_auffaellig"] = qi_calc.apply(eigene_bewertung, axis=1)

qi_calc["ist_N"] = qi_calc["QSErgBewStrukDialog"].astype(str).str.startswith("N")
qi_vergleich = qi_calc[~qi_calc["ist_N"]].copy()
qi_vergleich["offizielle_auffaellig"] = (
    ~qi_vergleich["QSErgBewStrukDialog"].astype(str).str.startswith("R")
).astype(int)

print(f"Vergleichbare Zeilen: {len(qi_vergleich):,}")
uebereinstimmung = (qi_vergleich["eigene_auffaellig"] == qi_vergleich["offizielle_auffaellig"]).mean()
print(f"Übereinstimmung eigene vs. offizielle Bewertung: {uebereinstimmung:.2%}")

Vergleichbare Zeilen: 110,975
Übereinstimmung eigene vs. offizielle Bewertung: 96.01%


In [9]:
kreuztabelle = pd.crosstab(
    qi_vergleich["eigene_auffaellig"], qi_vergleich["offizielle_auffaellig"],
    rownames=["eigene Bewertung"], colnames=["offizielle Bewertung"]
)
print(kreuztabelle)

offizielle Bewertung       0     1
eigene Bewertung                  
0                     100597     8
1                       4416  5954


**96,0 % Übereinstimmung auf Zeilenebene** — schon mit der simplen Punktwert-Regel kommen wir
ziemlich nah an IQTIGs Bewertung heran, aber eben nicht exakt. 4.416 Zeilen stuft unsere naive Regel
als auffällig ein, wo IQTIG „nicht auffällig" sagt (R10) — nur 8 Zeilen andersherum.

**Warum genau in diese eine Richtung so viele Abweichungen?**

## 5. Der Grund für die Abweichung: Konfidenzintervalle

`QS.Qualitätsindikator.csv` enthält noch zwei weitere Spalten, die unsere einfache Regel ignoriert
hat: `QSQI.KHVertrauensbereich` — das **Konfidenzintervall** des Hauses um seinen eigenen Messwert.
IQTIGs tatsächliche Regel ist kein reiner Punktvergleich, sondern ein **statistischer Test**: Ein Haus
gilt nur dann als auffällig, wenn der Referenzwert außerhalb seines Konfidenzintervalls liegt — liegt
der Referenzwert innerhalb (die Abweichung könnte also statistisches Rauschen sein), bleibt das Haus
unauffällig, selbst wenn der reine Punktwert über dem Referenzwert liegt.

In [10]:
def parse_bereich(s):
    try:
        lo, hi = str(s).split(" - ")
        return float(lo), float(hi)
    except Exception:
        return (np.nan, np.nan)

mismatch = qi_vergleich[
    (qi_vergleich["eigene_auffaellig"] == 1) & (qi_vergleich["offizielle_auffaellig"] == 0)
].copy()

bereiche = mismatch["QSQI.KHVertrauensbereich"].apply(parse_bereich)
mismatch["kh_lo"] = bereiche.apply(lambda t: t[0])
mismatch["kh_hi"] = bereiche.apply(lambda t: t[1])

im_intervall = (mismatch["referenzwert_num"] >= mismatch["kh_lo"]) & (mismatch["referenzwert_num"] <= mismatch["kh_hi"])
print(f"Anteil der {len(mismatch):,} Abweichungen, bei denen der Referenzwert im "
      f"Konfidenzintervall des Hauses liegt: {im_intervall.mean():.2%}")

Anteil der 4,416 Abweichungen, bei denen der Referenzwert im Konfidenzintervall des Hauses liegt: 99.46%


**99,5 % der Abweichungen** lassen sich so erklären: Der Referenzwert liegt im
Konfidenzintervall des Hauses — der Unterschied zum Punktwert ist statistisch nicht absicherbar, IQTIG
wertet das korrekterweise als „nicht auffällig", unsere naive Regel (die nur den Punktwert kennt)
fälschlich als „auffällig". Beispiel eines solchen Falls: Ergebnis 1,21, Referenzwert 1,10, aber
Konfidenzintervall 0,66–2,04 — der Referenzwert liegt mittendrin.

## 6. Hochrechnung auf Hausebene: Würde sich die Ziel-Variable ändern?

Dieselbe Frage jetzt für die eigentliche Ziel-Variable: Wenn wir `auffaellig_quote` und
`hat_viele_Probleme` mit der eigenen (naiven) Bewertung statt mit `QSErgBewStrukDialog` berechnen
würden — käme dieselbe Analysetabelle heraus?

In [11]:
qi_dedup = qi_calc.drop_duplicates(subset=["SO.QBID", "QSQI.Indikator"])
print(f"Nach Deduplizierung: {len(qi_dedup):,} Zeilen")

haus_selbst = qi_dedup.groupby("SO.QBID").agg(
    total_qi_selbst=("eigene_auffaellig", "count"),
    auffaellig_n_selbst=("eigene_auffaellig", "sum"),
).reset_index()
haus_selbst["auffaellig_quote_selbst"] = haus_selbst["auffaellig_n_selbst"] / haus_selbst["total_qi_selbst"]

print(f"Häuser mit mindestens einem selbst berechenbaren Indikator: {len(haus_selbst):,}")
print(f"Median auffaellig_quote (selbst berechnet): {haus_selbst['auffaellig_quote_selbst'].median():.2%}")
print(f"Ø Indikatoren pro Haus (selbst berechenbar): {haus_selbst['total_qi_selbst'].mean():.1f}")

Nach Deduplizierung: 45,308 Zeilen
Häuser mit mindestens einem selbst berechenbaren Indikator: 1,710
Median auffaellig_quote (selbst berechnet): 5.00%
Ø Indikatoren pro Haus (selbst berechenbar): 26.5


In [12]:
analysetabelle = pd.read_csv("../Data/analysetabelle.csv", low_memory=False)
print(f"Häuser in der offiziellen Analysetabelle: {len(analysetabelle):,}")
print(f"Median auffaellig_quote (offiziell): {analysetabelle['auffaellig_quote'].median():.2%}")
print(f"Ø Indikatoren pro Haus (offiziell, total_qi): {analysetabelle['total_qi'].mean():.1f}")

vergleich = haus_selbst.merge(
    analysetabelle[["SO.QBID", "auffaellig_quote", "total_qi", "hat_viele_Probleme"]],
    on="SO.QBID", how="inner"
)
print(f"\nGemeinsame Häuser: {len(vergleich):,} von {len(analysetabelle):,} "
      f"({len(vergleich)/len(analysetabelle):.1%})")

Häuser in der offiziellen Analysetabelle: 1,821
Median auffaellig_quote (offiziell): 5.88%
Ø Indikatoren pro Haus (offiziell, total_qi): 42.6

Gemeinsame Häuser: 1,710 von 1,821 (93.9%)


In [13]:
korrelation = vergleich["auffaellig_quote_selbst"].corr(vergleich["auffaellig_quote"])
print(f"Korrelation eigene vs. offizielle Haus-Quote: r = {korrelation:.3f}")

eigener_median = vergleich["auffaellig_quote_selbst"].median()
vergleich["hat_viele_selbst"] = (vergleich["auffaellig_quote_selbst"] > eigener_median).astype(int)
uebereinstimmung_gruppe = (vergleich["hat_viele_selbst"] == vergleich["hat_viele_Probleme"]).mean()
print(f"Übereinstimmung der Gruppenzuordnung (eigener Median-Split vs. offiziell): "
      f"{uebereinstimmung_gruppe:.1%}")

Korrelation eigene vs. offizielle Haus-Quote: r = 0.742
Übereinstimmung der Gruppenzuordnung (eigener Median-Split vs. offiziell): 75.9%


## 7. Fazit

| Vergleich | Ergebnis |
|---|---|
| Übereinstimmung auf Zeilenebene (nur berechenbare Teilmenge) | 96,0 % |
| Erklärung der Abweichungen | 99,5 % durch Konfidenzintervall des Hauses |
| Häuser überhaupt mit eigener Berechnung abdeckbar | 1.710 von 1.821 (93,9 %) |
| Ø Indikatoren pro Haus (eigen vs. offiziell) | 26,5 vs. 45,2 (–41 %) |
| Korrelation Haus-Quote (eigen vs. offiziell) | r = 0,74 |
| Übereinstimmung der Gruppenzuordnung (wenige/viele Probleme) | 75,9 % |

**Zwei unabhängige Gründe, warum eine Eigenberechnung schlechter wäre als die Übernahme von
`QSErgBewStrukDialog`:**

1. **Geringere Datenbasis:** Nur 36 % der Zeilen sind überhaupt selbst berechenbar (fehlender
   Referenzwert/Operator bei rund einem Drittel der Zeilen, dazu die aus Datenschutzgründen maskierten
   Kleinzahl-Ergebnisse). 111 Häuser (6,1 %) hätten gar keine einzige selbst berechenbare Zeile mehr —
   mehr als die 3 Häuser, die bei der offiziellen Methode verloren gehen.
2. **Fehlende Konfidenzintervall-Korrektur:** Eine naive Punktwert-vs-Referenzwert-Regel ignoriert,
   dass IQTIG die statistische Unsicherheit des Messwerts (`QSQI.KHVertrauensbereich`) explizit
   berücksichtigt. Genau das erklärt praktisch alle Abweichungen auf Zeilenebene.

**Schlussfolgerung:** Die im Hauptprojekt getroffene Entscheidung, `QSErgBewStrukDialog` direkt zu
übernehmen statt die Auffälligkeit selbst aus Ergebnis, Referenzwert und Operator nachzurechnen, war
richtig — IQTIGs fertige Bewertung ist vollständiger (mehr Häuser, mehr Indikatoren pro Haus) und
methodisch genauer (berücksichtigt Konfidenzintervalle), als es eine einfache Eigenberechnung leisten
könnte.